# Laboratoire 02 : Passerelle Python, Maîtrise de Pandas & Ingestion Bronze Multi-Sources
**Cours :** Business Intelligence et Analyse des Mégadonnées (`01 31 0367 00`)  
**Public cible :** Étudiants ingénieurs en 4e année B.Sc. (Génie Industriel et Management de la Production)  
**Durée :** 3 Heures Académiques (`135 min` / `2h15` de pratique guidée)  
**Prérequis :** Laboratoire 01 (`.venv` et échauffement Python validés)

---

## 🎯 1. Feuille de Route de la Séance (`2h15` Progressives)
Au **Labo 01**, vous avez pratiqué variables, listes, dictionnaires, boucles et fonctions Python. Ce TP transforme ces bases en véritable **chaîne d'ingénierie Data** sur des flux industriels :

1. **Module A — Diagnostic Étape 1 & Chemins Portables (~15 min) :** Vérification guidée de l'interpréteur `.venv`, explication ligne par ligne des mécanismes de découverte (`REPO_ROOT`) et procédures de dépannage.
2. **Module B — Nouveaux Idiomes Python pour Pipelines Data (~20 min) :** Maîtrisez les idiomes essentiels avant de lire les fichiers d'usine (`pathlib.Path`, gestionnaire de contexte `with open() as f:`, `json.load()`, compréhension de liste, tuples, `.items()`, `f-strings`).
3. **Module C — Pourquoi Pandas Domine la BI Industrielle (~30 min) :** Découvrez pourquoi on quitte Excel et les boucles Python (`Series` vs `DataFrame`, benchmark de vitesse vectorisée en direct, gardes `assert`, masques vectoriels `&` / `|` / `~`, sélection sûre `.loc[masque, cols]`, `NaN`, `.str`, `.value_counts()`, `.groupby()`).
4. **Module D — Ingestion Bronze Multi-Formats (~35 min) :** Chargez **Cas 0 EuroPack Beverage Bottling** (`CSV` + `JSON` + `XLSX` multi-onglets), attachez la traçabilité (`_ingestion_timestamp_utc`, `_source_file`), et exécutez l'exercice dédié **Cas 3 JSON imbriqué (`pd.json_normalize`)**.
5. **Module E — Staging Parquet Snappy & Vérification (~20 min) :** Sauvegardez en `.parquet` immuable (`index=False`), mesurez le gain d'espace disque et vérifiez la préservation du schéma. *(Le profilage qualité Bronze et la Scorecard démarrent le Labo 03 !)*

> **Jeu de Données Pédagogique vs Projets d'Équipe Notés :** Tous les travaux pratiques guidés (Labos 02–10) s'appuient sur le **Cas 0 (`EuroPack Beverage Bottling`)** — un jeu pédagogique neutre afin que chaque équipe maîtrise la chaîne complète sans divulguer les solutions des trois projets notés (**Cas 1 SmartPress**, **Cas 2 DataCo**, **Cas 3 Nexis**). L'Étape 10 entraîne également l'aplatissement JSON imbriqué du **Cas 3 (`Nexis Workforce`)** via `pd.json_normalize` et cartographie les trois packs de projet.


## ⚙️ 2. Contrôle de l'Environnement & Découverte Dynamique des Chemins
Avant de manipuler la donnée d'usine, un Ingénieur Data valide deux prérequis : **(1) le noyau Python `.venv` et ses bibliothèques**, puis **(2) des chemins portables** fonctionnant à l'identique sous Windows, macOS et Linux.

> **Guide de dépannage rapide si l'Étape 1 échoue :**
> - **`Python 3.11+ requis` ou `Bibliothèques manquantes` :** Cliquez sur **Select Kernel** (en haut à droite) → **Python Environments...** → **`.venv`**. S'il n'apparaît pas, lancez `setup_windows.ps1` / `./setup_macos.sh` / `./setup_linux.sh` dans le Terminal.
> - **`Impossible de localiser '03_labs_student_starters'` :** Utilisez **File → Open Folder...** dans VS Code pour ouvrir directement le dossier racine du dépôt (`ife_bi_bigdata_student` ou `IFE - BI and Big Data`), jamais un dossier parent.
> - **`Dossier de données brutes introuvable` :** Lancez `python verify_environment.py` dans le Terminal pour vérifier les jeux de données.

### Étape 1 : Contrôler les Bibliothèques, Comprendre la Découverte & Lister les Sources
Exécutez cette cellule telle quelle et lisez les commentaires Python pas à pas.

In [ ]:
# ==============================================================================
# ÉTAPE 1 : AUTO-DIAGNOSTIC DE L'ENVIRONNEMENT & DÉCOUVERTE DYNAMIQUE DU DÉPÔT
# ==============================================================================
# Lisez attentivement les commentaires ci-dessous : chaque ligne est expliquée pas à pas.
# En cas d'échec, le message affiché vous indiquera COMMENT corriger votre poste !

import sys  # Module standard indiquant quel interpréteur Python exécute ce notebook
from pathlib import Path  # Outil moderne pour gérer les chemins Windows / macOS / Linux

# ------------------------------------------------------------------------------
# 1A. CONTRÔLER LA VERSION DE PYTHON (3.11 OU SUPÉRIEURE REQUISE)
# ------------------------------------------------------------------------------
# sys.version_info est un tuple comme (3, 12, 4). Comparer les tuples teste majeur puis mineur.
print(f"Exécutable Python : {sys.executable}")
print(f"Version Python    : {sys.version.split()[0]}")

if sys.version_info < (3, 11):
    raise RuntimeError(
        f"Python 3.11+ est requis, mais Jupyter utilise {sys.version.split()[0]} ({sys.executable}).\n"
        "COMMENT CORRIGER DANS VS CODE :\n"
        "  1. Cliquez sur 'Select Kernel' (en haut à droite du notebook).\n"
        "  2. Choisissez 'Python Environments...' -> '.venv (Python 3.11+)'.\n"
        "  3. Si '.venv' n'existe pas encore, ouvrez un Terminal VS Code et lancez :\n"
        "     - Windows PowerShell : .\\setup_windows.ps1\n"
        "     - macOS Terminal     : ./setup_macos.sh\n"
        "     - Linux Terminal     : ./setup_linux.sh"
    )

# ------------------------------------------------------------------------------
# 1B. CONTRÔLER LA PRÉSENCE DES BIBLIOTHÈQUES ESSENTIELLES DANS CE NOYAU
# ------------------------------------------------------------------------------
# Si vous avez choisi le Python global au lieu de '.venv', ce bloc le détecte aussitôt !
_required_libs = {
    "pandas": "Manipulation de tables DataFrame & lecture CSV/JSON/Excel",
    "openpyxl": "Lecture des classeurs Excel multi-onglets (.xlsx)",
    "pyarrow": "Écriture des fichiers colonnaires Parquet compressés Snappy",
}
_missing_libs = []
for _pkg_name, _purpose in _required_libs.items():
    try:
        # __import__() importe dynamiquement une bibliothèque à partir de son nom texte
        _mod = __import__(_pkg_name)
        print(f"  [OK] {_pkg_name:<10} v{getattr(_mod, '__version__', 'inconnue')} -> {_purpose}")
    except ImportError:
        _missing_libs.append(f"{_pkg_name} ({_purpose})")

if _missing_libs:
    raise ImportError(
        "Bibliothèques manquantes dans le noyau Jupyter actif :\n  - "
        + "\n  - ".join(_missing_libs)
        + f"\nInterpréteur actif : {sys.executable}\n"
        + "COMMENT CORRIGER :\n"
        + "  1. Cause n°1 : mauvais noyau ! Passez sur '.venv' via 'Select Kernel' (en haut à droite).\n"
        + "  2. Si '.venv' est bien sélectionné, ouvrez un Terminal avec (.venv) actif et exécutez :\n"
        + "     pip install -r requirements.txt"
    )

# Maintenant que l'environnement est sain, importons le reste des outils
import json
import time
from datetime import datetime, timezone
import pandas as pd
from IPython.display import display  # Affichage interactif en tableau HTML sous Jupyter/VS Code

# ------------------------------------------------------------------------------
# 1C. LOCALISER AUTOMATIQUEMENT LA RACINE DU DÉPÔT (AUCUN CHEMIN FIGÉ C:\ !)
# ------------------------------------------------------------------------------
# Pourquoi un underscore '_' devant '_cwd' ou '_cand' ?
# Convention Python signalant une variable temporaire locale utilisée uniquement pour le repérage.
# Path.cwd().resolve() récupère le dossier courant exact dans lequel tourne le notebook.
_cwd = Path.cwd().resolve()
REPO_ROOT = None

# Que signifie `*_cwd.parents` ?
# `_cwd.parents` liste tous les dossiers parents en remontant ([lab_02, 03_labs, racine, ...]).
# L'étoile `*` DÉPAQUETTE cette liste à la suite de `_cwd` pour tout parcourir dans une seule boucle.
for _cand in [_cwd, *_cwd.parents]:
    # Cas 1 : Sommes-nous déjà dans le dépôt étudiant / enseignant ?
    if (_cand / "03_labs_student_starters").exists():
        REPO_ROOT = _cand
        break  # On arrête dès qu'on trouve '03_labs_student_starters'
    # Cas 2 : Et si VS Code avait ouvert le dossier parent contenant le clone ?
    for _sub in ("ife_bi_bigdata_course", "ife_bi_bigdata_student"):
        if (_cand / _sub / "03_labs_student_starters").exists():
            REPO_ROOT = _cand / _sub
            break
    if REPO_ROOT is not None:
        break

# `assert condition, message` stoppe proprement avec une explication claire si REPO_ROOT reste None
assert REPO_ROOT is not None, (
    f"Impossible de localiser '03_labs_student_starters' depuis {_cwd}.\n"
    "COMMENT CORRIGER DANS VS CODE :\n"
    "  Allez dans File -> Open Folder... et ouvrez directement la racine du dépôt "
    "(ex. 'ife_bi_bigdata_student' dans Courses/), puis rouvrez ce notebook."
)

# ------------------------------------------------------------------------------
# 1D. CONFIGURER LE CAS PÉDAGOGIQUE (CAS 0) & LES DOSSIERS SANDBOX
# ------------------------------------------------------------------------------
# Tous les Labos guidés 02-10 utilisent le Cas 0 (EuroPack Beverage Bottling — jeu pédagogique neutre).
# Votre projet noté d'équipe porte sur l'un des Cas 1, 2 ou 3 dans votre espace projet !
CASE_NAME = "case0_europack_bottling"
# Jeux de données des projets notés d'équipe (hors TP guidés) :
# CASE_NAME = "case1_smartpress_automotive"
# CASE_NAME = "case2_dataco_supply_chain"
# CASE_NAME = "case3_nexis_workforce"

# L'opérateur '/' sur Path gère automatiquement '/' (macOS/Linux) et '\' (Windows) !
STUDENT_BRONZE_DIR = REPO_ROOT / "03_labs_student_starters" / "lab_02_bronze_ingestion_and_profiling" / "data" / CASE_NAME
REF_CASE_DATA = REPO_ROOT / "05_case_studies_reference_solutions" / CASE_NAME / "data"

# Expression ternaire (`A if condition else B`) :
# Fonctionne aussi bien sur le dépôt étudiant (data/ dans lab_02) que sur le dépôt enseignant (05_case_studies...).
BRONZE_RAW = STUDENT_BRONZE_DIR if STUDENT_BRONZE_DIR.exists() else (REF_CASE_DATA / "bronze")
SANDBOX_DATA = REPO_ROOT / "03_labs_student_starters" / "sandbox_outputs" / CASE_NAME
BRONZE_STAGED = SANDBOX_DATA / "bronze_staged"

# mkdir(parents=True, exist_ok=True) crée l'arborescence sans planter si le dossier existe déjà
BRONZE_STAGED.mkdir(parents=True, exist_ok=True)


def resolve_stage_dir(stage_name: str) -> Path:
    """Utilisée aux Labos 03-10 pour localiser les Parquet amont dans sandbox_outputs."""
    cand = SANDBOX_DATA / stage_name
    # `any(...)` vérifie qu'au moins un fichier *.parquet existe dans cand
    if cand.exists() and any(cand.glob("*.parquet")):
        return cand
    ref_cand = REF_CASE_DATA / stage_name
    if ref_cand.exists():
        return ref_cand
    raise FileNotFoundError(
        f"Étape '{stage_name}' introuvable dans {cand}. Exécutez d'abord le laboratoire précédent."
    )


print("\nRacine du dépôt :", REPO_ROOT)
print("Dossier Bronze  :", BRONZE_RAW, "| Existe :", BRONZE_RAW.exists())
print("Zone de Staging :", BRONZE_STAGED)

assert BRONZE_RAW.exists(), (
    f"Dossier de données brutes introuvable : {BRONZE_RAW}\n"
    "COMMENT CORRIGER : Lancez 'python verify_environment.py' dans le Terminal VS Code."
)

print("\nFichiers sources détectés :")
for _p in sorted(BRONZE_RAW.iterdir()):
    if _p.is_file():
        # Formatage : `:<32` aligne le texte à gauche sur 32 car. ; `,.1f` affiche 1 décimale
        print(f"  - {_p.name:<32} ({_p.stat().st_size / 1024:,.1f} Ko)")


## 🌉 3. Passerelle Python (Après le Labo 01) : Idiomes des Pipelines Data
Le Labo 01 couvrait variables, listes, dictionnaires, boucles `for` et fonctions. L'ingestion industrielle mobilise **quelques tournures Python compactes** présentes du Labo 02 au Labo 10. Apprivoisons-les une micro-cellule à la fois !

### Étape 2 : Chemins (`pathlib.Path`), Gestionnaire de Contexte (`with open`) & `json.load()`
Pourquoi utilise-t-on `with open(...) as f:` plutôt qu'un ancien `f = open(...)` ?
- Le **gestionnaire de contexte (`with ... as ...:`)** garantit la **fermeture automatique du fichier** dès la sortie du bloc indenté—même si une erreur survient pendant la lecture.
- Notez aussi `encoding="utf-8"` : imposer UTF-8 évite les corruptions d'accents européens (`é`, `ł`, `€`) entre Windows (`cp1252`) et macOS/Linux.
- À l'intérieur du bloc, `json.load(f)` transforme le texte JSON directement en **liste Python de dictionnaires** (déjà vue au Labo 01 !).

In [ ]:
# L'objet Path inspecte les métadonnées d'un fichier sans même l'ouvrir
sample_json_path = BRONZE_RAW / "stoppage_logs.json"
print("Nom du fichier :", sample_json_path.name)
print("Extension      :", sample_json_path.suffix)
print("Dossier parent :", sample_json_path.parent.name)

# Syntaxe : `with open(chemin, mode, encoding) as alias:`
with open(sample_json_path, "r", encoding="utf-8") as f:
    preview_logs = json.load(f)

print(f"Type chargé    : {type(preview_logs).__name__} contenant {len(preview_logs)} enregistrements")

# À FAIRE : Inspectez le DERNIER journal d'arrêt (indice -1) via preview_logs[-1] et affichez ses clés (.keys())
last_log = None  # REMPLACER PAR : preview_logs[-1]
if last_log is not None:
    print("Clés du dernier arrêt :", list(last_log.keys()))
    print("Contenu du dernier arrêt :", last_log)
else:
    print("⏳ [À FAIRE] Assignez last_log = preview_logs[-1] puis ré-exécutez cette cellule.")


### Étape 3 : Compréhensions de Liste, Tuples, Dictionnaires `.items()` & `f-strings`
Quatre raccourcis Python reviennent constamment aux Étapes 7 à 13 :
1. **Compréhension de liste :** `[expr for x in seq if cond]` filtre et construit une liste en une ligne lisible (à la place d'une boucle + `.append()`).
2. **Tuple `(a, b, c)` :** Séquence immuable idéale pour regrouper des listes fixes de tables ou d'extensions.
3. **Dictionnaire `.items()` :** `for cle, val in mon_dict.items():` récupère **ensemble la clé et la valeur** à chaque tour.
4. **`f-strings` formatées :** `f"{val:,}"` sépare les milliers et `f"{nom:<20}"` aligne les colonnes.

In [ ]:
# 1. Boucle classique Labo 01 vs Compréhension de liste moderne
all_files = sorted(BRONZE_RAW.iterdir())             # Liste triée des fichiers bruts du dossier
tabular_names_loop = []                              # Liste vide pour collecter les noms retenus
for p in all_files:                                  # Parcourt chaque objet Path du dossier
    if p.suffix in (".csv", ".json", ".xlsx"):       # Vérifie l'extension via un tuple immuable
        tabular_names_loop.append(p.name)            # Ajoute le nom du fichier (sans le chemin)

# Compréhension de liste équivalente en 1 ligne :
tabular_names_comp = [p.name for p in all_files if p.suffix in (".csv", ".json", ".xlsx")]  # Même filtre en 1 ligne
assert tabular_names_loop == tabular_names_comp      # Vérifie que les deux méthodes sont identiques
print("Fichiers sources détectés :", tabular_names_comp)

# 2. Parcours d'un dictionnaire avec .items() + alignement f-string
expected_formats = {                                 # Associe chaque format à son rôle industriel
    "CSV": "Relevés PLC d'embouteillage toutes les 15 min",
    "JSON": "Journaux d'arrêts ligne GMAO semi-structurés",
    "XLSX": "Référentiels ERP multi-onglets (Lines, Shifts, Bottle_Formats)",
}
for fmt_key, description in expected_formats.items():  # Dépaquette clé et valeur à chaque tour
    print(f"  {fmt_key:<5} -> {description}")            # Aligne la clé à gauche sur 5 caractères

# À FAIRE : Écrivez une compréhension de liste gardant UNIQUEMENT les fichiers '.csv' ou '.json' (sans '.xlsx')
stream_files = None  # REMPLACER PAR : [p.name for p in all_files if p.suffix in (".csv", ".json")]
if stream_files is not None:                         # Garde : ne teste qu'une fois le TODO rempli
    assert len(stream_files) == 2 and all(name.endswith((".csv", ".json")) for name in stream_files)  # 2 fichiers attendus
    print("Fichiers flux hors Excel :", stream_files)
else:
    print("⏳ [À FAIRE] Complétez la compréhension de liste pour stream_files.")


## 🐼 4. Introduction à Pandas : Pourquoi Toute la Chaîne BI S'y Appuie
Du Labo 02 au Labo 04, nous travaillerons intensivement avec **`pandas`** avant de construire nos schémas en étoile et vues SQL dans **DuckDB** (Labos 05–06) puis nos tableaux de bord visuels (Labos 07–09). Pourquoi abandonner Excel ou les boucles Python ?

| Capacité | Excel / Tableurs | Python Pur (`list[dict]`) | **`pandas` (`DataFrame`)** |
| :--- | :--- | :--- | :--- |
| **Volumétrie** | Plafond strict `1 048 576` lignes ; ralentit > 200k | Limité par la RAM, boucles lentes | Gère des millions de lignes via tableaux C/NumPy |
| **Exécution** | Recalcul cellule par cellule | Élément par élément en boucle `for` | **Vectorisation** : calcule toute une colonne d'un coup (10×–100× plus rapide) |
| **Schéma & Nuls** | Inverse dates JJ/MM vs MM/JJ ; blancs ambigus | Conversions manuelles `float()` | `.dtypes` explicites (`float64`, `datetime64`), propagation native `NaN` |
| **Boîte à Outils Data** | Pas de jointures SQL, regex ni Parquet | Tout coder soi-même | Connecteurs natifs `.read_csv/json/excel/parquet`, `.str`, `.dt`, `.groupby`, `.merge` |

### Étape 4 : Benchmark en Direct — Boucle Python vs Vectorisation Pandas
Pandas repose sur deux briques : **`pd.Series`** (une colonne typée avec son index) et **`pd.DataFrame`** (une table 2D alignant plusieurs `Series`). Nous allons comparer une boucle Python pure et Pandas sur **100 000 créneaux d'embouteillage simulés** à partir de trois variables initiales :
- `n_rows` : Nombre de créneaux PLC de 15 minutes simulés (`100 000`).
- `bottled_list` : Nombre total de bouteilles remplies par créneau (`800`–`849` bouteilles).
- `rejected_list` : Nombre de bouteilles rejetées (fissures/sous-remplissage) par créneau (`8`–`14`).

**Partie 4A — `zip()` et Boucle `for` en Python Pur :** Observez d'abord ce que produit `zip(bottled_list, rejected_list)` puis mesurez le temps de la boucle ligne par ligne.

In [ ]:
# Simulation de 100 000 relevés haute fréquence sous forme de listes Python
n_rows = 100_000                                       # Nombre total de créneaux simulés
bottled_list = [800 + (i % 50) for i in range(n_rows)] # Bouteilles remplies par créneau (800..849)
rejected_list = [8 + (i % 7) for i in range(n_rows)]   # Bouteilles rejetées par créneau (8..14)

# Aperçu de ce que zip() associe sur les 5 premiers éléments des deux listes
zipped_preview = list(zip(bottled_list[:5], rejected_list[:5]))  # Liste de tuples (bottled, rejected)
print("5 premières paires zip(bottled, rejected) :", zipped_preview)

# 1. Boucle Python ligne par ligne (approche Labo 01)
t0 = time.perf_counter()                               # Démarre le chronomètre haute précision
reject_pct_py = []                                     # Liste vide pour stocker les taux de rejet
for btl, rej in zip(bottled_list, rejected_list):      # Parcourt une paire (bottled, rejected) à la fois
    reject_pct_py.append((rej / btl) * 100.0)          # Calcule le % de rejet et l'ajoute
py_ms = (time.perf_counter() - t0) * 1000              # Durée écoulée en millisecondes
print(f"Boucle Python pure (100k lignes) : {py_ms:6.2f} ms")


**Partie 4B — Inspection d'un `DataFrame` Pandas & Benchmark Vectorisé :** Chargez maintenant ces mêmes listes dans un `pd.DataFrame`, inspectez ses dimensions, ses types de colonnes et ses 5 premières lignes (`.head()`), puis mesurez la vitesse vectorisée !

In [ ]:
# 2. Arithmétique vectorisée Pandas (approche Labos 02-04)
df_bench = pd.DataFrame({"bottled": bottled_list, "rejected": rejected_list})  # Crée une table 2D

t1 = time.perf_counter()                                 # Démarre le chronomètre pour Pandas
df_bench["reject_pct"] = (df_bench["rejected"] / df_bench["bottled"]) * 100.0  # Calcul vectorisé sur toute la colonne
pd_ms = (time.perf_counter() - t1) * 1000                # Durée écoulée en millisecondes

print(f"Vectorisation Pandas (100k)      : {pd_ms:6.2f} ms  -> Gain : {py_ms / max(pd_ms, 0.01):.1f}x plus rapide !")
print("\nComment inspecter un DataFrame :")
print("  Dimensions (.shape)  ->", df_bench.shape, "(lignes, colonnes)")
print("  Colonnes (.columns)  ->", list(df_bench.columns))
print("  Types (.dtypes)      ->", df_bench.dtypes.to_dict())
print("  Series vs DataFrame  ->", type(df_bench).__name__, "vs", type(df_bench["reject_pct"]).__name__)
print("\n5 premières lignes via display(df_bench.head(5)) :")
display(df_bench.head(5))


### Étape 5 : Gardes `assert` / `is not None` & Masques Booléens Vectoriels (`&`, `|`, `~`)
**Contexte Industriel :** Dans le hall d'embouteillage EuroPack, les capteurs de pression de soutireuse enregistrent parfois des valeurs physiquement impossibles (`-99.0 bar` lors d'une déconnexion ou `85.0 bar` sur un pic de signal — plage normale `[0.0, 12.0] bar`) ou des événements d'un banc d'essai fantôme non référencé (`LINE_Z`). Avant de calculer les KPI d'usine, nous construisons des **masques booléens vectoriels** sur des `pd.Series` pour isoler ces lignes défectueuses sans boucle lente.

Deux réflexes de robustesse protègent vos notebooks d'ingénierie :
- **Gardes `if df is not None:` et `assert` :** Toujours vérifier qu'un objet est bien chargé avant d'appeler `.shape` ou `.to_parquet()`.
- **Python `and`/`or`/`not` vs Pandas `&`/`|`/`~` :**
  - Utilisez **`and`, `or`, `not`** pour tester **une seule** valeur scalaire (`if x > 0 and y < 10:`).
  - Utilisez les opérateurs bit-à-bit **`&` (ET), `|` (OU), `~` (NON) AVEC PARENTHÈSES** sur des **colonnes entières Pandas** : `(s < 0.0) | (s > 12.0)`.
  - *Pourquoi les parenthèses ?* Sans `( ... )`, Python évalue `|` avant `<` et échoue avec `TypeError` !

In [ ]:
# Condition scalaire Python -> utilise `and` / `not`
boiler_temp_c = 71.5                                   # Température unique du pasteurisateur (°C)
interlock_ok = True                                    # État du verrouillage de sécurité
scalar_safe = (boiler_temp_c <= 80.0) and interlock_ok # Combine deux booléens scalaires avec `and`
print("Contrôle scalaire validé :", scalar_safe)

# Colonne Pandas (Series) -> EXIGE `&` (ET), `|` (OU), `~` (NON) + parenthèses !
sample_pressures = pd.Series([4.2, -99.0, 5.1, 85.0, 4.8])  # Relevés simulés de pression (bar)
valid_lines = {"LINE_A", "LINE_B"}                          # Ensemble des lignes valides de l'ERP
sample_ids = pd.Series(["LINE_A", "LINE_Z", "LINE_B", "LINE_A", "LINE_B"])  # IDs journalisés

# Signaler une pression hors [0.0, 12.0] bar OU une ligne absente du catalogue
bad_pressure_mask = (sample_pressures < 0.0) | (sample_pressures > 12.0)  # True pour valeurs aberrantes
orphan_line_mask = ~sample_ids.isin(valid_lines)                          # '~' inverse : True pour IDs inconnus

# À FAIRE : Combinez bad_pressure_mask OU orphan_line_mask via '|'
any_anomaly_mask = None  # REMPLACER PAR : bad_pressure_mask | orphan_line_mask
if any_anomaly_mask is not None:                       # Garde contre None avant .sum()
    assert int(any_anomaly_mask.sum()) == 2, f"2 anomalies attendues, obtenu {int(any_anomaly_mask.sum())}"  # Compte les True
    print("Masque combiné des anomalies :", any_anomaly_mask.tolist())    # Affiche le masque en liste Python
    print("Lignes signalées :")
    display(pd.DataFrame({"line_id": sample_ids, "pressure_bar": sample_pressures})[any_anomaly_mask])  # Filtre via [masque] et affiche
else:
    print("⏳ [À FAIRE] Combinez bad_pressure_mask | orphan_line_mask ci-dessus.")


### Étape 6 : Bac à Sable des Outils Pandas (`.loc[]`, `.isna()`, `.str`, `.value_counts()`, `.groupby()`)
Avant de manipuler nos `23 164` relevés PLC, testez ces 5 opérations phares sur un mini-extrait de 6 lignes d'embouteillage :
- **`.loc[filtre_lignes, [colonnes]]`** (*label-based location*, par opposition à `.iloc` par position entière) : Sélection propre sans ambiguïté ni avertissement de copie.
- **`.isna().sum()` :** Pourquoi `.sum()` pour *compter* les valeurs manquantes ? `.isna()` renvoie des booléens `True`/`False`, et en Python/Pandas `True == 1` et `False == 0`—`.sum()` additionne donc les `1` pour compter les `NaN` (`NaN` n'est *pas* zéro !).
- **`.str.strip().str.upper()` & `.assign()` :** Simuler la normalisation en mémoire *sans modifier* `mini_plant` (`.assign(col=...)` renvoie une nouvelle copie avec la colonne ajoutée).
- **`df['col']`, `.value_counts(dropna=False)` & `.groupby(...)[col].sum()` :** Les crochets `[...]` après un `DataFrame` (ou un `GroupBy`) sélectionnent une colonne `Series` (ou filtrent les lignes via un masque booléen).

In [ ]:
mini_plant = pd.DataFrame({
    "line_id": ["Line_A ", "LN-B", "LINE_A", "line_c", "LINE_Z", "LINE_B"],
    "filler_pressure_bar": [4.4, -99.0, 4.7, 4.1, 85.0, 5.2],
    "pasteurizer_temp_C": [72.1, 73.4, None, 71.8, 75.0, 72.9],
    "bottled_units": [980, 820, 1010, 950, 40, 845],
})

print("1. Valeurs manquantes par colonne (True=1, False=0 -> .sum() compte les NaN) :")
print(mini_plant.isna().sum().to_string())

print("\n2. Aperçu chaînes normalisées (df['col'] sélectionne une Series ; .str applique les méthodes texte) :")
clean_names = mini_plant["line_id"].str.strip().str.upper()  # Nouvelle Series sans modifier mini_plant
print(clean_names.value_counts().to_string())

# À FAIRE A : Utilisez .loc[masque, cols] (sélection par label) pour extraire ['line_id', 'filler_pressure_bar'] si filler_pressure_bar < 0 OU > 12
mini_outliers = None  # REMPLACER PAR : mini_plant.loc[(mini_plant["filler_pressure_bar"] < 0) | (mini_plant["filler_pressure_bar"] > 12), ["line_id", "filler_pressure_bar"]]

# À FAIRE B : .assign(clean_id=...) crée une copie avec la colonne ; ['bottled_units'] choisit la colonne à sommer
mini_grouped = None  # REMPLACER PAR : mini_plant.assign(clean_id=clean_names).groupby("clean_id")["bottled_units"].sum()

if mini_outliers is not None and mini_grouped is not None:
    assert len(mini_outliers) == 2
    print("\nValeurs extrêmes détectées via .loc[] :")
    display(mini_outliers)
    print("\nBouteilles produites par ligne normalisée :")
    display(mini_grouped.to_frame("bottled_units"))
else:
    print("\n⏳ [À FAIRE] Complétez mini_outliers (.loc[]) et mini_grouped (.groupby()) ci-dessus.")


## 📥 5. Partie 1 : Ingestion du Flux CSV de Télémesure & Traçabilité d'Audit
En **Bronze**, chaque table doit rester un **miroir fidèle sans perte** enrichi de deux colonnes de lignage :
- `_ingestion_timestamp_utc` : horodatage ISO-8601 UTC (`datetime.now(timezone.utc).isoformat()`) indiquant *quand le pipeline a capturé le fichier* (distinct de l'horodatage capteur `timestamp` !).
- `_source_file` : nom exact du fichier source (`"line_telemetry.csv"`).

### Étape 7 : Charger `line_telemetry.csv` & Vérifier Schéma et Mémoire
Complétez les `# À FAIRE` pour lire `BRONZE_RAW / csv_name` dans `df_telemetry` et attacher les deux colonnes d'audit.

> **Pourquoi `.info()` affiche-t-il `Dtype = object` pour `timestamp` ET `_ingestion_timestamp_utc` ?**  
> Dans Pandas 2.x, les colonnes texte (`str`) sont stockées comme des tableaux `object`. Appeler `.isoformat()` convertit `datetime.now(timezone.utc)` en **chaîne ISO-8601** (`"2026-...+00:00"`), donc les *deux* colonnes contiennent du texte (`type(val) == str`). Vous pouvez vérifier le vrai type Python via `type(df[col].iloc[0])`. La conversion vers `datetime64` via `pd.to_datetime()` est volontairement différée au **Labo 03 (Silver)** afin de garder le Bronze strictement brut.

In [ ]:
csv_name = "line_telemetry.csv"
now_utc = datetime.now(timezone.utc).isoformat()

# À FAIRE : Charger le CSV via pd.read_csv(BRONZE_RAW / csv_name)
df_telemetry = None  # REMPLACER PAR : pd.read_csv(BRONZE_RAW / csv_name)

# À FAIRE : Ajouter _ingestion_timestamp_utc = now_utc et _source_file = csv_name

if df_telemetry is not None:
    assert {"_ingestion_timestamp_utc", "_source_file"}.issubset(df_telemetry.columns), "Ajoutez les 2 colonnes d'audit !"
    print(f"Chargement {csv_name} -> {df_telemetry.shape[0]:,} lignes x {df_telemetry.shape[1]} colonnes")
    print("\nRésumé schéma & RAM via .info(memory_usage='deep') :")
    df_telemetry.info(memory_usage="deep")
    print("\nAperçu des 3 premières lignes :")
    display(df_telemetry.head(3))
else:
    print("⏳ [À FAIRE] Chargez df_telemetry et ajoutez les deux colonnes d'audit ci-dessus.")


### Étape 8 : Première Exploration de la Télémesure (Plage Temporelle, Formats & Statistiques)
Observez dans `.info()` que `timestamp` est chargé comme chaîne texte—un CSV ne conserve pas de type `datetime` natif ! Inspectez les bornes `min`/`max`, la répartition de `bottle_format` et le résumé statistique `.describe()`.

In [ ]:
if df_telemetry is not None:
    # À FAIRE : Calculez min/max de 'timestamp', value_counts() de 'bottle_format' et .describe().round(2)
    ts_min = None         # REMPLACER PAR : df_telemetry["timestamp"].min()
    ts_max = None         # REMPLACER PAR : df_telemetry["timestamp"].max()
    format_counts = None  # REMPLACER PAR : df_telemetry["bottle_format"].value_counts(dropna=False)

    if ts_min is not None and ts_max is not None and format_counts is not None:
        print(f"Plage brute timestamp : {ts_min}  -->  {ts_max}  (repérez l'anomalie 2099-* au sommet !)")
        print("\nRépartition des formats de bouteilles (bottle_format) :")
        print(format_counts.to_string())
        print("\nStatistiques descriptives (.describe()) :")
        print(df_telemetry[["ambient_temp_C", "pasteurizer_temp_C", "filler_pressure_bar", "valve_cycles_k", "bottled_units", "rejected_units"]].describe().round(2).to_string())
    else:
        print("⏳ [À FAIRE] Complétez ts_min, ts_max et format_counts ci-dessus.")
else:
    print("⏳ Complétez d'abord l'Étape 7 afin que df_telemetry soit chargé.")


## 📋 6. Partie 2 : Ingestion des Journaux GMAO Semi-Structurés (`stoppage_logs.json`)
Les outils GMAO exportent les ordres d'arrêt ligne sous forme de tableaux d'objets JSON.

### Étape 9 : Parser le JSON Plat via Gestionnaire de Contexte (`with open`) -> `df_stoppages`
Appliquez le modèle de l'Étape 2 : ouvrez `BRONZE_RAW / json_name`, décodez `raw_records = json.load(f)`, construisez `pd.DataFrame(raw_records)` et ajoutez `_ingestion_timestamp_utc` ainsi que `_source_file`.

In [ ]:
json_name = "stoppage_logs.json"

# À FAIRE : Ouvrez BRONZE_RAW / json_name avec `with open(..., "r", encoding="utf-8") as f:`,
# chargez raw_records via json.load(f), créez df_stoppages = pd.DataFrame(raw_records) et ajoutez les colonnes d'audit.
df_stoppages = None  # REMPLACER PAR VOTRE CODE

if df_stoppages is not None:
    assert {"_ingestion_timestamp_utc", "_source_file"}.issubset(df_stoppages.columns)
    print(f"Chargement {json_name} -> {df_stoppages.shape[0]:,} arrêts x {df_stoppages.shape[1]} colonnes")
    print("Colonnes :", list(df_stoppages.columns))
    print("\nAperçu des 5 premiers journaux d'arrêt :")
    display(df_stoppages.head(5))
else:
    print("⏳ [À FAIRE] Chargez df_stoppages et ajoutez les colonnes d'audit ci-dessus.")


### Étape 10 (Compétence Projet — JSON Imbriqué) : Aplatissement avec `pd.json_normalize`
Le flux pédagogique guidé traite le **Cas 0 (`EuroPack Beverage Bottling`)**, où `stoppage_logs.json` est déjà plat. Dans le **Cas 3 (`Nexis Workforce`)**, `workforce_events.json` encapsule plusieurs attributs dans un sous-objet imbriqué **`details`** (dont le champ médical sensible RGPD/PII `details.medical_note`).

> **Repères rapides des 3 Projets d'Équipe Notés :**
> - **Cas 1 (`SmartPress Automotive`) :** `sensor_telemetry.csv` (SCADA presses hydrauliques 15 min), `maintenance_logs.json` (tickets GMAO pannes presses) et `machines_and_shifts.xlsx` (`Machines`, `Shifts`, `Product_Variants`).
> - **Cas 2 (`DataCo Hydraulics`) :** `order_items_trans.csv` (lignes de commande ERP), `shipment_events.json` (jalons WMS/3PL portant `warehouse_hub`, `shipping_mode`, `carrier_code`, `freight_cost_eur`, `sla_penalty_eur`) et `master_catalog.xlsx` (`Products`, `Customers`, `Warehouses`, `Carrier_SLA_Policies`).
> - **Cas 3 (`Nexis Workforce`) :** `shift_production_performance.csv` (compteurs MES incluant les intérimaires anonymes `spot_fill` sur `268/720` quarts), `workforce_events.json` (pointages des `165` salariés badgés avec sous-objet `details`) et `workforce_roster_and_skills.xlsx` (`Employees`, `Skill_Matrix`, `Incentive_Policies`, `Shift_Staffing_Plan`).

Comparez `pd.DataFrame(raw)` et `pd.json_normalize(raw, sep="_")` sur **`workforce_events.json` (Cas 3)** sans écraser vos variables du Cas 0 :

In [ ]:
c3_student = REPO_ROOT / "03_labs_student_starters" / "lab_02_bronze_ingestion_and_profiling" / "data" / "case3_nexis_workforce" / "workforce_events.json"
c3_ref = REPO_ROOT / "05_case_studies_reference_solutions" / "case3_nexis_workforce" / "data" / "bronze" / "workforce_events.json"
c3_json_path = c3_student if c3_student.exists() else c3_ref

if c3_json_path.exists():
    with open(c3_json_path, "r", encoding="utf-8") as f:
        c3_raw_events = json.load(f)

    # Un pd.DataFrame naïf laisse 'details' piégé sous forme de dict Python dans la colonne :
    df_c3_naive = pd.DataFrame(c3_raw_events[:5])
    print("Colonnes avec pd.DataFrame naïf :", list(df_c3_naive.columns))
    print("Exemple de cellule 'details' :", df_c3_naive["details"].iloc[0])

    # À FAIRE : Utilisez pd.json_normalize(c3_raw_events, sep="_") puis listez les colonnes 'details_*'
    df_c3_flat = None  # REMPLACER PAR : pd.json_normalize(c3_raw_events, sep="_")
    if df_c3_flat is not None:
        detail_cols = sorted(c for c in df_c3_flat.columns if c.startswith("details_"))
        print(f"\npd.json_normalize(..., sep='_') -> {df_c3_flat.shape[0]:,} lignes x {df_c3_flat.shape[1]} colonnes")
        print(f"Colonnes 'details_*' aplaties ({len(detail_cols)}) :", detail_cols)
        print("Attention : 'details_medical_note' est une donnée médicale sensible (PII) à purger en Silver !")
    else:
        print("\n⏳ [À FAIRE] Assignez df_c3_flat = pd.json_normalize(c3_raw_events, sep='_') pour voir l'aplatissement.")


## 📊 7. Partie 3 : Ingestion du Classeur Excel Multi-Onglets
Pourquoi les référentiels ERP arrivent-ils dans 3 onglets distincts de `plant_master_data.xlsx` ?
Parce que **`Lines`**, **`Shifts`** et **`Bottle_Formats`** sont des entités métiers indépendantes dotées de leurs propres clés primaires (futures tables de **Dimensions Kimball** au Labo 05). Remarquez comment l'itération sur un **tuple** `(df_lines, df_shifts, df_bottle_formats)` évite de dupliquer le code d'audit !

### Étape 11 : Lister les Onglets Excel & Charger les Trois Tables Référentielles

In [ ]:
excel_name = "plant_master_data.xlsx"
excel_path = BRONZE_RAW / excel_name

# Inspection préalable des onglets via pd.ExcelFile(...)
sheet_names = pd.ExcelFile(excel_path).sheet_names
print("Onglets Excel détectés :", sheet_names)

# À FAIRE : Chargez chaque onglet via pd.read_excel(excel_path, sheet_name="...")
df_lines = None          # REMPLACER PAR VOTRE CODE
df_shifts = None         # REMPLACER PAR VOTRE CODE
df_bottle_formats = None # REMPLACER PAR VOTRE CODE

# À FAIRE : Décommentez la boucle tuple ci-dessous une fois les 3 tables chargées :
# for df_ref in (df_lines, df_shifts, df_bottle_formats):
#     df_ref["_ingestion_timestamp_utc"] = now_utc
#     df_ref["_source_file"] = excel_name

if all(df is not None for df in (df_lines, df_shifts, df_bottle_formats)):
    assert len(df_lines) == 4 and len(df_shifts) == 3 and len(df_bottle_formats) == 3
    print(f"\nLines ({df_lines.shape[0]} lignes) :")
    display(df_lines)
    print(f"\nShifts ({df_shifts.shape[0]} lignes) :")
    display(df_shifts)
    print(f"\nBottle_Formats ({df_bottle_formats.shape[0]} lignes) :")
    display(df_bottle_formats)
else:
    print("⏳ [À FAIRE] Chargez les 3 onglets Excel et ajoutez leurs métadonnées d'audit.")


## 💾 8. Partie 4 : Exportation au Format Colonnaire Parquet Snappy (`index=False`)
Pourquoi convertir CSV/JSON/Excel en **Apache Parquet (`.parquet`)** ?
- **Stockage colonnaire :** DuckDB et Pandas ne lisent que les colonnes demandées sans balayer tout le fichier.
- **Schéma intégré :** Préserve fidèlement `int64`, `float64` et textes sans ré-inférence hasardeuse.
- **Compression Snappy + `index=False` :** Réduit nettement la taille disque tout en évitant d'injecter une colonne parasite `__index_level_0__`.

### Étape 12 : Exporter les 5 Tables Bronze par Itération `.items()` sur Dictionnaire

In [ ]:
# Correspondance nom cible -> DataFrame en mémoire
outputs = {
    "bronze_telemetry.parquet": df_telemetry,
    "bronze_stoppages.parquet": df_stoppages,
    "bronze_lines.parquet": df_lines,
    "bronze_shifts.parquet": df_shifts,
    "bronze_bottle_formats.parquet": df_bottle_formats,
}

# Garde défensive : vérifie que les Étapes 7, 9 et 11 ont toutes été complétées
missing_tables = [fname for fname, df_obj in outputs.items() if df_obj is None]
assert not missing_tables, (
    f"Export impossible — tables encore à None : {missing_tables}. "
    "Complétez d'abord les Étapes 7, 9 et 11 ci-dessus !"
)

# À FAIRE : Parcourez `outputs.items()`, exportez chaque table via `index=False, compression="snappy"`,
# et affichez le volume de lignes et la taille en Ko.
for fname, df_out in outputs.items():
    out_path = BRONZE_STAGED / fname
    # REMPLACER `pass` PAR : df_out.to_parquet(out_path, index=False, compression="snappy")
    pass
    if out_path.exists():
        print(f"  [OK] {fname:<34} {df_out.shape[0]:>7,} lignes -> {out_path.stat().st_size / 1024:>8.2f} Ko")


### Étape 13 (Clôture) : Vérifier l'Aller-Retour Parquet (`Zéro Fuite d'Index`) & Gain de Compression
Relisez `bronze_telemetry.parquet` pour confirmer l'absence de `__index_level_0__`, la préservation du schéma et comparez la taille totale brute vs Parquet Snappy.

> **Vérification Finale :** Cliquez sur **Restart Kernel and Run All Cells** avant de terminer le Labo 02 ! Suite au **Labo 03** : Profilage Qualité Bronze & Nettoyage Silver Partie 1.

In [ ]:
raw_kb = sum(p.stat().st_size for p in BRONZE_RAW.iterdir() if p.is_file()) / 1024
staged_files = sorted(BRONZE_STAGED.glob("*.parquet"))
staged_kb = sum(p.stat().st_size for p in staged_files) / 1024

if staged_files:
    df_roundtrip = pd.read_parquet(BRONZE_STAGED / "bronze_telemetry.parquet")
    assert "__index_level_0__" not in df_roundtrip.columns, "Ré-exportez avec index=False !"
    # À FAIRE : Calculez compression_ratio = raw_kb / staged_kb et affichez le bilan disque
    print(f"Total brut (CSV/JSON/XLSX) : {raw_kb:,.1f} Ko")
    print(f"Total Parquet Snappy       : {staged_kb:,.1f} Ko  ({len(staged_files)} fichiers)")
    print("Nombre de colonnes typées :", len(df_roundtrip.dtypes))
    print("\n🎉 Labo 02 terminé ! Lancez 'Restart Kernel and Run All Cells' puis passez au Labo 03.")
else:
    print("⏳ Complétez d'abord l'Étape 12 (.to_parquet) pour créer les fichiers sur disque.")
